In [1]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import random

In [4]:
data = pd.read_csv(("C:/Users/Neezy/Videos/Attacks/data/numerical.csv"))
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [5]:
X = data.iloc[:, :-1].values
y = data.iloc[:, -1].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
classifier = DecisionTreeClassifier(random_state=42)
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy * 100:.2f}%')

Accuracy: 97.57%


In [20]:
importances = classifier.feature_importances_
feature_imp_df = pd.DataFrame({'Feature': data.columns[:-1], 'Gini Importance': importances}).sort_values(
    'Gini Importance', ascending=False)

important_features = []
importances_to_numpy = feature_imp_df.to_numpy()
for i in range(0,100):
     important_features.append(importances_to_numpy[i][0])

In [21]:
important_features.append('label')

In [22]:
df = data[important_features]

In [23]:
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
classifier = DecisionTreeClassifier(random_state=42)
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy * 100:.2f}%')

Accuracy: 97.73%


In [24]:
def make_pred(sample): 
    single_pred = classifier.predict([sample])
    return single_pred 

In [25]:
ransomware_training_sample_indices = np.where(y_train==1)
idx = np.random.choice(ransomware_training_sample_indices[0], 500, replace=False)
xtrain_poison_candidates = X_train[idx] 

In [26]:
print(len(xtrain_poison_candidates))

500


In [ ]:
def make_initial_adversarial(sampled_row,perturbation):
    lst = sampled_row.tolist().copy() #removed [0]
    print(f"initial sample: {lst}") 
    adv_indices = [i for i, x in enumerate(lst) if x == 0] # Track Indices
    print(f"Identified features to perturb: {len(adv_indices)}") 

    adversarial_sample = list(map(lambda x: x+perturbation if x==0 else x, lst))
    print(f"perturbed sample: {adversarial_sample}")
    prediction = make_pred(adversarial_sample) 
        
    if prediction == 0: 
        print("Initial perturbation successful") 
        return adversarial_sample, adv_indices
    else: 
        print("perturbation failed")
        return None, None

def decrement_list(lst, perturbation, adversarial_indices):
    lst2 = lst.copy()
    final_adv_indices = [] # find all the features that have been perturbed.  
    print(f"Identified indices to check if perturbation was necessary: {len(adversarial_indices)}")
    
    for j in adversarial_indices: 
        lst2[j] -= perturbation 
        check_pred = make_pred(lst2) 
        if check_pred != 0:
            lst2[j] += perturbation
            final_adv_indices.append(j)  
    
    print(f"{len(adversarial_indices) - len(final_adv_indices)} features were identified as uneccessary to perturb: features to be optimized: {len(final_adv_indices)} features")        
    
    print(f"Semi Optimized sample: {lst2}")
    return lst2, final_adv_indices ## Return final indices

def optimize(sample, final_adversarial_sample,final_adv_indices): 
                
    print(f"final adversarial indices to check: {len(final_adv_indices)}")
    for j in final_adv_indices: ## trace all the features that removing the perturbation caused incorrect prediction
        while final_adversarial_sample[j]>sample[j]: 
            final_adversarial_sample[j] -= 1
            if make_pred(final_adversarial_sample) == 1: 
                final_adversarial_sample[j] +=1
                break
                
                    
    print(f"Final Adversarial sample: {final_adversarial_sample} ")
    return final_adversarial_sample
              

In [28]:
def poison(perturbation): 
    samples = xtrain_poison_candidates
    adversarial_samples = []
    for i, sample in enumerate(samples):  
        
        adv = make_initial_adversarial(sample,perturbation)
        if adv is not None:
           adversarial_sample = decrement_list(adv, perturbation)
           final_adversarial_sample = optimize(sample, perturbation, adversarial_sample)
           l2 = np.linalg.norm(final_adversarial_sample-sample)
           print(f"l2 distance: {l2}")
           adversarial_samples.append(final_adversarial_sample)      
           print(f"sample number: {i}/{len(samples)} complete")
    
    print()
    return adversarial_samples

perturbation=25
asamples = poison(perturbation)

initial sample: [4, 0, 4, 0, 0, 0, 16, 3, 3, 0, 0, 0, 0, 0, 0, 25, 0, 15, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 19, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 5, 0, 2, 0, 0, 2, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
perturbed sample: [29, 25, 29, 25, 25, 25, 41, 28, 28, 25, 25, 25, 25, 25, 25, 50, 25, 40, 27, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 44, 25, 26, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 26, 25, 25, 25, 26, 25, 25, 26, 30, 25, 27, 25, 25, 27, 25, 25, 25, 25, 28, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 28, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25]
Initial perturbation successful
Semi Optimized sample: [29, 0, 29, 0, 0, 0, 41, 28, 28, 0, 0, 0, 0, 25, 0, 50, 0, 40, 27, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 44, 0, 26, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 26, 0, 0, 0, 26, 0, 0, 26, 30, 0, 27, 0, 0, 27, 0, 0, 0

In [29]:
len(asamples)

434

In [30]:
np.save("dt_adversarial_samples_top_100.npy", np.array(asamples))

In [31]:
mask = np.ones(X_train.shape[0], dtype=bool)
mask[idx] = False

X_train = X_train[mask]
y_train = y_train[mask]

print(len(X_train))

1966


In [32]:
remove_samples_from_asamples = random.sample(asamples, k=126)
print(len(remove_samples_from_asamples))

126


In [33]:
result = [item for item in asamples if item not in remove_samples_from_asamples]
np_result = np.array(result)
print(len(result))
poisoned_xtrain = np.concatenate((X_train, np_result))
len(poisoned_xtrain)

193


2159

In [34]:
adv_y = []
for i in range(0,193): 
    adv_y.append(1) 
    
print(len(adv_y))

np_advy = np.array(adv_y)
poisoned_ytrain = np.concatenate((y_train, np_advy))
print(len(poisoned_ytrain)) 

193
2159


In [35]:
poisoned_classifier = DecisionTreeClassifier(random_state=41)
poisoned_classifier.fit(poisoned_xtrain, poisoned_ytrain)
y_pred_poisoned = poisoned_classifier.predict(X_test)
poisoned_accuracy = accuracy_score(y_test, y_pred_poisoned)
print(f'Accuracy: {poisoned_accuracy * 100:.2f}%')

Accuracy: 97.73%


In [36]:
def make_pred_poisoned(samples):
    predictions = [] 
    for sample in samples:
        single_pred = poisoned_classifier.predict([sample])
        predictions.append(single_pred)
        print(single_pred) 
        if single_pred[0] == 1: 
            print("Correctly identified adversarial ransomware sample") 
        else: 
            print("Adversarial sample still adversarial")

    zero_counter = 0 
    one_counter = 0 
    for pred in predictions:
        if pred != 1: 
            zero_counter +=1 
        else: 
            one_counter+=1 
        
    
    print(f"Samples still adversarial: {zero_counter}")
    print(f"adversarial_samples identified: {one_counter}")

make_pred_poisoned(remove_samples_from_asamples)

[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correc